# **Universal YOLOv5 Training (ISL)**
### Works on: Google Colab & Kaggle Kernels
### Architecture: YOLOv5 Medium (v5m)
### **Note for Kaggle Users:** Enable 'Internet Access' in Settings to clone the repo.

In [ ]:
# Cell 1: Environment Setup & Install
import os
import sys

# Detect Environment
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = 'kaggle_web_client' in os.environ

if IS_COLAB:
    ROOT_DIR = '/content'
    print("✅ Detected: Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')
elif IS_KAGGLE:
    ROOT_DIR = '/kaggle/working'
    print("✅ Detected: Kaggle Kernel")
else:
    ROOT_DIR = '.'

# Setup Paths
YOLO_DIR = os.path.join(ROOT_DIR, 'yolov5')

# Clone YOLOv5 Repo
if not os.path.exists(YOLO_DIR):
    print("Cloning YOLOv5 Repository...")
    !git clone https://github.com/ultralytics/yolov5 "{YOLO_DIR}"
    
# Install Requirements
!pip install -qr "{YOLO_DIR}/requirements.txt"

import torch
print(f"\n🔥 GPU Available: {torch.cuda.is_available()}")

---

In [ ]:
# Cell 2: Data Loading (Universal)

# === METHOD A: Roboflow ===
# !pip install roboflow -q
# from roboflow import Roboflow
# rf = Roboflow(api_key="KEY")
# project = rf.workspace().project()
# dataset = project.version(1).download("yolov5")
# dataset_dir = dataset.location

# === METHOD B: Manual Zip / Drive ===
zip_path = ""  # <--- PASTE PATH (e.g., /content/drive/MyDrive/ISL.zip)
dataset_dir = os.path.join(ROOT_DIR, "dataset_isl")

if zip_path and os.path.exists(zip_path):
    import shutil
    if os.path.exists(dataset_dir): 
        shutil.rmtree(dataset_dir)
    os.makedirs(dataset_dir, exist_ok=True)
    print("Extracting dataset...")
    !unzip -q "{zip_path}" -d "{dataset_dir}"
    print(f"✅ Extracted to {dataset_dir}")
else:
    if 'dataset_dir' not in locals():
        dataset_dir = os.path.join(ROOT_DIR, "dataset_isl")
        print("⚠️ No zip provided. Assuming dataset exists at: " + dataset_dir)

In [ ]:
# Cell 3: Generate data.yaml (Auto-Config)
# This creates the config file tailored to the current environment path
import yaml

# EDIT THIS LIST for your specific ISL Classes
class_names = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 
               'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 
               'U', 'V', 'W', 'X', 'Y', 'Z', 'space', 'nothing'] 

data_config = {
    'path': dataset_dir,        # Root dir
    'train': 'train/images',    # Relative to root
    'val': 'valid/images',      # Relative to root
    'nc': len(class_names),
    'names': class_names
}

yaml_path = os.path.join(YOLO_DIR, 'data_isl.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print(f"✅ Config created at: {yaml_path}")

---

In [ ]:
# Cell 4: Train YOLOv5m
# Using batch 16 to prevent Out-Of-Memory on free GPUs
import os

# Change directory to yolov5 so train.py runs correctly
os.chdir(YOLO_DIR)

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 50 \
    --data data_isl.yaml \
    --weights yolov5m.pt \
    --name isl_model \
    --cache

In [ ]:
# Cell 5: Export Weights
best_path = os.path.join(YOLO_DIR, 'runs/train/isl_model/weights/best.pt')

if IS_COLAB:
    dest = '/content/drive/MyDrive/ISL_YOLOv5m_Final.pt'
    if os.path.exists(best_path):
        import shutil
        shutil.copy(best_path, dest)
        print(f"✅ Saved to Google Drive: {dest}")
else:
    print(f"✅ Training Done. Download from: {best_path}")
    # Note for Kaggle: The file is in /kaggle/working/yolov5/runs/...